In [1]:
import numpy as np
from skimage.measure import regionprops
from model_ranking.utils import load_h5
from collections.abc import Iterable

from typing import Any, List, Dict
from numpy.typing import NDArray

INFO: P [MainThread] 2026-03-02 11:16:09,377 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calculate_extents(lbl, func=np.median):
    """Aggregate bounding box sizes of objects in label images."""
    if (isinstance(lbl, np.ndarray) and lbl.ndim == 4) or (
        not isinstance(lbl, np.ndarray) and isinstance(lbl, Iterable)
    ):
        return func(
            np.stack([calculate_extents(_lbl, func) for _lbl in lbl], axis=0), axis=0
        )

    n = lbl.ndim
    assert n in (2, 3), ValueError("label image should be 2- or 3-dimensional")

    regs = regionprops(lbl)
    if len(regs) == 0:
        return np.zeros(n)
    else:
        extents = np.array([np.array(r.bbox[n:]) - np.array(r.bbox[:n]) for r in regs])
        if func is None:
            return extents
        else:
            return func(extents, axis=0)

# BBBC039

In [3]:
from pytorch3dunet.datasets.dsb import TIF_txt_Dataset

In [4]:
transformer_config: Dict[str, Any] = {
    "raw": [
        {"name": "ToTensor", "expand_dims": True},
    ],
    "label": [
        {"name": "Relabel"},
        {"name": "BlobsToMask", "append_label": True},
        {"name": "ToTensor", "expand_dims": True},
    ]
}

In [5]:
dataset = TIF_txt_Dataset(
    image_dir="/g/kreshuk/talks/data/BBBC039/images",
    mask_dir="/g/kreshuk/talks/data/BBBC039/instance_annotations/instance_labels",
    phase='train',
    transformer_config=transformer_config,
    filenames_path="/g/kreshuk/talks/data/BBBC039/test.txt",
    global_norm=False,
)

In [6]:
obj_sizes = []
for i in range(len(dataset)):
    raw, label = dataset[i]
    per_slice_objs = calculate_extents(label[1].numpy().squeeze().astype(np.uint8), func=None)
    obj_sizes.append(per_slice_objs)
objects = np.concatenate(obj_sizes, axis=0)
print(objects.shape)

(5765, 2)


In [7]:
print(np.median(objects, axis=0), np.std(objects, axis=0))
print(np.min(objects, axis=0), np.max(objects, axis=0))
print(np.mean(objects, axis=0))

[30. 30.] [8.40169728 8.1084926 ]
[3 4] [62 68]
[29.61665221 29.66556808]


### Sample n object

In [8]:
# Sample n objects from the objects array and print mean object size
n = 50
sampled_indices = np.random.choice(objects.shape[0], n, replace=False)
sampled_objects = objects[sampled_indices]

print(f"Sampled {n} objects")
print(f"Mean object size: {np.mean(sampled_objects, axis=0)}")

Sampled 50 objects
Mean object size: [31.52 29.86]


### median per image

In [10]:
object_sizes = np.zeros((len(dataset), 2))
for i in range(len(dataset)):
    raw, label = dataset[i]
    extents = calculate_extents(label[1].numpy().squeeze().astype(np.uint8))
    object_sizes[i] = extents


In [11]:
print(np.median(object_sizes, axis=0), np.std(object_sizes, axis=0))
print(np.min(object_sizes, axis=0), np.max(object_sizes, axis=0))
print(np.mean(object_sizes, axis=0))

[29.75 29.5 ] [1.62416512 1.44922752]
[24. 27.] [34.5 33. ]
[29.61458333 29.6875    ]


# Hoechst

In [12]:
from pytorch3dunet.datasets.dsb import Hoechst_Dataset

In [13]:
transformer_config: Dict[str, Any] = {
    "raw": [

        {"name": "ToTensor", "expand_dims": True},
    ],
    "label": [
        {"name": "Relabel"},
        {"name": "BlobsToMask", "append_label": True},
        {"name": "ToTensor", "expand_dims": True},
    ]
}

In [14]:
dataset = Hoechst_Dataset(
    image_dir="/g/kreshuk/talks/data/Hoechst/test_nuclei/images/png",
    mask_dir="/g/kreshuk/talks/data/Hoechst/test_nuclei/annotations",
    phase='train',
    transformer_config=transformer_config,
    global_norm=False,
)

In [15]:
obj_sizes = []
for i in range(len(dataset)):
    raw, label = dataset[i]
    per_slice_objs = calculate_extents(label[1].numpy().squeeze().astype(np.uint8), func=None)
    obj_sizes.append(per_slice_objs)
objects = np.concatenate(obj_sizes, axis=0)
print(objects.shape)

(438, 2)


In [16]:
print(np.median(objects, axis=0), np.std(objects, axis=0))
print(np.min(objects, axis=0), np.max(objects, axis=0))
print(np.mean(objects, axis=0))

[44. 47.] [18.75189655 18.45827593]
[3 3] [105 103]
[42.9109589  44.36986301]


### Sample n objects

In [18]:
# Sample n objects from the objects array and print mean object size
n = 50
sampled_indices = np.random.choice(objects.shape[0], n, replace=False)
sampled_objects = objects[sampled_indices]

print(f"Sampled {n} objects")
print(f"Mean object size: {np.mean(sampled_objects, axis=0)}")

Sampled 50 objects
Mean object size: [42.46 43.56]


### Median per image

In [19]:
object_sizes = np.zeros((len(dataset), 2))
for i in range(len(dataset)):
    raw, label = dataset[i]
    extents = calculate_extents(label[1].numpy().squeeze().astype(np.uint8))
    object_sizes[i] = extents


In [20]:
print(np.median(object_sizes, axis=0), np.std(object_sizes, axis=0))
print(np.min(object_sizes, axis=0), np.max(object_sizes, axis=0))
print(np.mean(object_sizes, axis=0))

[44. 47.] [5.51565953 5.01597448]
[41. 38.] [59.5 59. ]
[45.45 46.8 ]


# S_BIAD895

In [21]:
from pytorch3dunet.datasets.dsb import Standard_TIF_Dataset

In [22]:
transformer_config: Dict[str, Any] = {
    "raw": [
        {"name": "ToTensor", "expand_dims": True},
    ],
    "label": [
        {"name": "Relabel"},
        {"name": "BlobsToMask", "append_label": True},
        {"name": "ToTensor", "expand_dims": True},
    ]
}

In [23]:
dataset = Standard_TIF_Dataset(
    image_dir="/g/kreshuk/talks/data/S-BIAD895/ZeroCostDL4Mic/Stardist_v2/Stardist/Test/Raw",
    mask_dir="/g/kreshuk/talks/data/S-BIAD895/ZeroCostDL4Mic/Stardist_v2/Stardist/Test/Masks",
    phase='train',
    transformer_config=transformer_config,
    global_norm=False,
)

In [24]:
obj_sizes = []
for i in range(len(dataset)):
    raw, label = dataset[i]
    per_slice_objs = calculate_extents(label[1].numpy().squeeze().astype(np.uint8), func=None)
    obj_sizes.append(per_slice_objs)
objects = np.concatenate(obj_sizes, axis=0)
print(objects.shape)

(435, 2)


In [25]:
print(np.median(objects, axis=0), np.std(objects, axis=0))
print(np.min(objects, axis=0), np.max(objects, axis=0))
print(np.mean(objects, axis=0))

[25. 25.] [5.66650766 5.97695668]
[4 4] [44 46]
[24.03218391 24.09655172]


### sample n objects

In [26]:
# Sample n objects from the objects array and print mean object size
n = 50
sampled_indices = np.random.choice(objects.shape[0], n, replace=False)
sampled_objects = objects[sampled_indices]

print(f"Sampled {n} objects")
print(f"Mean object size: {np.mean(sampled_objects, axis=0)}")

Sampled 50 objects
Mean object size: [24.46 23.84]


### median per slice

In [27]:
object_sizes = np.zeros((len(dataset), 2))
for i in range(len(dataset)):
    raw, label = dataset[i]
    extents = calculate_extents(label[1].numpy().squeeze().astype(np.uint8))
    object_sizes[i] = extents


In [28]:
print(np.median(object_sizes, axis=0), np.std(object_sizes, axis=0))
print(np.min(object_sizes, axis=0), np.max(object_sizes, axis=0))
print(np.mean(object_sizes, axis=0))

[24.5 24.5] [0.5 0.5]
[24. 24.] [25. 25.]
[24.5 24.5]


# S_BIAD634

In [29]:
transformer_config: Dict[str, Any] = {
    "raw": [
        {"name": "ToTensor", "expand_dims": True},
    ],
    "label": [
        {"name": "Relabel"},
        {"name": "BlobsToMask", "append_label": True},
        {"name": "ToTensor", "expand_dims": True},
    ]
}

In [30]:
dataset = TIF_txt_Dataset(
    image_dir="/g/kreshuk/talks/data/S-BIAD634/dataset/rawimages",
    mask_dir="/g/kreshuk/talks/data/S-BIAD634/dataset/groundtruth",
    phase='train',
    transformer_config=transformer_config,
    filenames_path="/g/kreshuk/talks/data/S-BIAD634/dataset/test.txt",
    global_norm=False,
)

<tifffile.TiffFile 'otherspecimen_3.tif'> shaped series shape does not match page shape


In [31]:
obj_sizes = []
for i in range(len(dataset)):
    raw, label = dataset[i]
    per_slice_objs = calculate_extents(label[1].numpy().squeeze().astype(np.uint8), func=None)
    obj_sizes.append(per_slice_objs)
objects = np.concatenate(obj_sizes, axis=0)
print(objects.shape)

(3395, 2)


In [32]:
print(np.median(objects, axis=0), np.std(objects, axis=0))
print(np.min(objects, axis=0), np.max(objects, axis=0))
print(np.mean(objects, axis=0))

[70. 70.] [279.04559636 274.07315994]
[2 2] [1023 1334]
[214.53637703 195.39499264]


### sample n object

In [41]:
# Sample n objects from the objects array and print mean object size
n = 50
sampled_indices = np.random.choice(objects.shape[0], n, replace=False)
sampled_objects = objects[sampled_indices]

print(f"Sampled {n} objects")
print(f"Mean object size: {np.mean(sampled_objects, axis=0)}")

Sampled 50 objects
Mean object size: [181.92 167.14]
